# Building a modeling table in SQL

Data Scientist track, lesson DS3. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** build a modeling table with one row for each person from event tables, with features measured as of a cutoff and a label from after it, remove retry copies, count only what the app knew, and reconcile the table to its source.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

SEED = 2026       # the seed of the whole Data Scientist track


def show(sql, limit=12):
    """Run a query and print its rows as text: the column names, then one line for each row."""
    cur = con.execute(sql)
    rows = cur.fetchall()
    print(" | ".join(column[0] for column in cur.description))
    for row in rows[:limit]:
        print(" | ".join("NULL" if value is None else str(value) for value in row))
    if len(rows) > limit:
        print(f"... {len(rows)} rows in all")


def rows(sql):
    """Run a query and return its rows as lists, with numbers as numbers."""
    return [[float(v) if hasattr(v, "as_tuple") else v for v in row] for row in con.execute(sql).fetchall()]

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)
print("Data loaded: 21,841 people and 177,379 events.")

## The query as one statement

The next lessons build the table of this lesson with one `WITH` statement. This cell stores it as `ONE_STATEMENT`; in unit 5 you compare your pieces with it. Run it as it is.

In [ ]:
ONE_STATEMENT = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""      # the whole query as one statement: what the notebooks of the next lessons run

## Unit 1: One row per person, and a cutoff for each

**Predict first.** The query joins `users` to the signup event of each user. Will it return fewer rows than there are users in the window, the same number, or more?

In [ ]:
con.execute("""CREATE OR REPLACE VIEW people AS
SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
       s.event_ts AS signup_ts,
       s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
       s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
FROM users AS u
JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'""")
show("""SELECT COUNT(*) AS people, COUNT(DISTINCT user_id) AS distinct_people,
       (SELECT COUNT(*) FROM users WHERE signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31') AS users_in_window
FROM people""")

## Unit 2: Count only what the app knew

**Predict first.** Which rule takes more events out of the first week: removing the retry copies, or keeping only what had arrived by the cutoff? Roughly how many events does each take?

In [ ]:
con.execute("""CREATE OR REPLACE VIEW clean AS
SELECT *
FROM events
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1""")
con.execute("""CREATE OR REPLACE VIEW known AS
SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
FROM people AS p
JOIN clean AS c ON c.user_id = p.user_id
WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff""")
show("""SELECT
  (SELECT COUNT(*) FROM people AS p JOIN events AS e ON e.user_id = p.user_id WHERE e.event_ts <= p.cutoff) AS happened_by_cutoff,
  (SELECT COUNT(*) FROM people AS p JOIN clean AS c ON c.user_id = p.user_id WHERE c.event_ts <= p.cutoff) AS without_copies,
  (SELECT COUNT(*) FROM known) AS known_at_cutoff""")
show("""SELECT (SELECT COUNT(*) FROM events) - (SELECT COUNT(*) FROM clean) AS copies_removed""")

## Unit 3: Features as counts of what was known

**Predict first.** How many of the 17,816 people have no visit and no expense in their first 7 days: about 5 percent, about a quarter, or about half?

In [ ]:
con.execute("""CREATE OR REPLACE VIEW features AS
SELECT user_id,
       COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
       COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
       COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
       COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
       CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
       CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
       CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
FROM known
GROUP BY user_id""")
show("""SELECT COUNT(*) AS people, COUNT(*) FILTER (WHERE active_days_7d = 0) AS with_no_activity,
       SUM(recent_3d) AS recent_events, SUM(sessions_7d + expenses_7d) AS week_events
FROM features""")

## Unit 4: The label comes from the other side

**Predict first.** Count only `add_expense` as activity instead of both events. Will the share of people who churned go up, stay about the same, or go down?

In [ ]:
con.execute("""CREATE OR REPLACE VIEW label AS
SELECT p.user_id,
       CAST(NOT EXISTS (SELECT 1 FROM events AS e
                        WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                          AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
FROM people AS p""")
show("""SELECT ROUND(100.0 * AVG(churned), 1) AS churn_both_events FROM label""")
for names in ("('session_start', 'add_expense')", "('session_start')", "('add_expense')"):
    print(names, rows(f"""SELECT ROUND(100.0 * AVG(CAST(NOT EXISTS (SELECT 1 FROM events AS e
        WHERE e.user_id = p.user_id AND e.event_name IN {names} AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER)), 1) FROM people AS p""")[0][0])

## Unit 5: Reconcile before anyone fits it

**Predict first.** The second route removes the copies in pandas and applies the same two clocks. Will its totals of visits and expenses equal the SQL totals exactly?

In [ ]:
con.execute("""CREATE OR REPLACE VIEW modeling_table AS
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id""")
# check 1: one row per person, none lost
show("""SELECT COUNT(*) AS table_rows, COUNT(DISTINCT user_id) AS distinct_people, (SELECT COUNT(*) FROM people) AS people FROM modeling_table""")
# check 2: the same totals from a second route, in pandas, straight from the raw events
ev = con.sql("SELECT user_id, event_ts, event_name, platform, received_at FROM events").df()
pe = con.sql("SELECT user_id, cutoff FROM people").df()
ev = ev.drop_duplicates(["user_id", "event_ts", "event_name", "platform"]).merge(pe, on="user_id", validate="many_to_one")
ev = ev[(ev["event_ts"] <= ev["cutoff"]) & (ev["received_at"] <= ev["cutoff"])]
print(int((ev["event_name"] == "session_start").sum()), int((ev["event_name"] == "add_expense").sum()))
show("""SELECT SUM(sessions_7d) AS sessions, SUM(expenses_7d) AS expenses FROM modeling_table""")
# check 3: each signup month, with its share of the table
show("""SELECT month(signup_date) AS month, COUNT(*) AS people, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_table,
       ROUND(100.0 * AVG(churned), 1) AS pct_churned
FROM modeling_table GROUP BY month ORDER BY month""")
# check 4: hand the table to pandas; the next lessons build the same table with one statement
table = con.sql("SELECT * FROM modeling_table ORDER BY user_id").df()
print(table.shape, table.equals(con.sql(ONE_STATEMENT).df()))

## Finish it

### Exercise 1: Finish it

**Task:** **How fast do people start?** The starter holds a query, `first_expense`, with the time of each person's first known expense (NULL when there is none). Write a query over it and put the list [people whose first expense came within an hour of signup, people with an expense in the first week] in `answer_1`.

**Expected:** `answer_1` is a list of two whole numbers.

In [ ]:
first_expense = """SELECT p.user_id, p.signup_ts, MIN(k.event_ts) FILTER (WHERE k.event_name = 'add_expense') AS first_expense
FROM people AS p JOIN known AS k ON k.user_id = p.user_id GROUP BY p.user_id, p.signup_ts"""      # given: the time of each person's first known expense, NULL when there is none
answer_1 = None  # replace None with [people whose first expense came within an hour of signup, people with an expense in the week]: rows(...)[0] of a query over first_expense

In [ ]:
_ = check("Exercise 1", "answer_1", [10466, 12386], traps=[([10466, 17816], "The second number counts every person, including those with no expense. Count the first_expense column, not the rows: COUNT of a column skips NULL.")])

<details>
<summary>Hint</summary>

Wrap the query in parentheses after `FROM`. `COUNT(*) FILTER (WHERE first_expense <= signup_ts + INTERVAL 1 HOUR)` counts the people within the hour; for the second number use a count that skips `NULL`.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = rows(f"""SELECT COUNT(*) FILTER (WHERE ...), COUNT(...)
FROM ({first_expense})""")[0]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_1 = rows(f"""SELECT COUNT(*) FILTER (WHERE first_expense <= signup_ts + INTERVAL 1 HOUR), COUNT(first_expense)
FROM ({first_expense})""")[0]
```

10,466 people started within an hour, out of 12,386 with an expense in the week. `COUNT(*)` would have said 17,816, every person, because it counts rows and a person with no expense is still a row; `COUNT(first_expense)` skips the `NULL`s. A feature built on a wrong denominator would call everyone without an expense slow.

</details>

## On your own

### Exercise 2: On your own

**Task:** **How many people does the arrival rule touch?** Before you start, say in one sentence what the result looks like: one whole number, much smaller than 17,816. For each person, count the visits and expenses that had happened by the cutoff in `clean` (copies removed, arrival ignored), and compare with `sessions_7d + expenses_7d` in `features`. Put the number of people whose two counts differ in `answer_2`.

**Expected:** `answer_2` is a whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 46, traps={318: "That counts against the raw events, with the retry copies still in. Use clean, so that only the arrival rule differs."})

<details>
<summary>Hint</summary>

Join `people` to `clean` on `user_id`, keep the rows with `event_ts <= cutoff` and the two event names, count them for each person, then join to `features` and compare.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"One whole number, a few dozen at most."*

```python
answer_2 = rows("""WITH raw_week AS (SELECT p.user_id, COUNT(*) AS n FROM people AS p JOIN clean AS c ON ... GROUP BY p.user_id)
SELECT COUNT(*) FROM raw_week AS r JOIN features AS f ON ... WHERE ...""")[0][0]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = rows("""WITH raw_week AS (
  SELECT p.user_id, COUNT(*) AS n
  FROM people AS p JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.event_name IN ('session_start', 'add_expense')
  GROUP BY p.user_id)
SELECT COUNT(*) FROM raw_week AS r JOIN features AS f ON f.user_id = r.user_id
WHERE r.n <> f.sessions_7d + f.expenses_7d""")[0][0]
```

46 people have a different count when the arrival rule is dropped, 0.26 percent of the table: a late event matters for a few people only. Against the raw events, with the copies in, the count is 318: a different question.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague builds the label for the people who signed up from May 15 to May 31, but measures the end of the window from the cutoff: `e.event_ts <= p.cutoff + INTERVAL 30 DAY`. Change the end of the window to the horizon of `people` and put the number of churned people among these signups in `answer_3`.

**Expected:** `answer_3` is a whole number.

In [ ]:
flawed = rows("""SELECT COUNT(*) FILTER (WHERE churned = 1)
FROM (SELECT p.user_id,
             CAST(NOT EXISTS (SELECT 1 FROM events AS e
                              WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                                AND e.event_ts > p.cutoff AND e.event_ts <= p.cutoff + INTERVAL 30 DAY) AS INTEGER) AS churned
      FROM people AS p WHERE p.signup_date >= DATE '2026-05-15')""")[0][0]        # colleague: churned people among the May 15 to May 31 signups
answer_3 = flawed

In [ ]:
_ = check("Exercise 3", "answer_3", 1665, traps={1613: "That window runs 30 days past the cutoff, to day 37, so it counts activity that the definition leaves out. End the window at the horizon."})

<details>
<summary>Hint</summary>

The label is defined as no activity in days 8 to 30 after signup. Which column of `people` already holds the end of day 30?

</details>

<details>
<summary>Half-solution</summary>

```python
... AND e.event_ts <= p.horizon) AS INTEGER) AS churned
```

</details>

<details>
<summary>Solution</summary>

```python
answer_3 = rows("""SELECT COUNT(*) FILTER (WHERE churned = 1)
FROM (SELECT p.user_id,
             CAST(NOT EXISTS (SELECT 1 FROM events AS e
                              WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                                AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
      FROM people AS p WHERE p.signup_date >= DATE '2026-05-15')""")[0][0]
```

With the horizon, 1,665 of these 2,758 people churned. The window of the colleague runs to day 37, past the end of the data for anyone who signed up after May 24, and it counts a week of activity that the definition leaves out, so it finds 1,613: fewer churners, because a longer window finds more people active. The error is silent: the query runs and the numbers look plausible.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")